# 🔬 Faster R-CNN — Esophageal Cancer Detection
### 50 Epochs | Fast | Loss Curves | Confusion Matrix | Metrics Table | best.pt

In [ ]:
import os
os.system('pip install pycocotools -q')
print('✅ Done')

In [ ]:
import json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict
import torch, torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')
print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only!"}')

In [ ]:
# ═══ CONFIG ← only change COCO_ROOT if needed ═════════════
COCO_ROOT   = '/kaggle/input/datasets/rugvedkulkarni250809/esophageal-balanced-coco/coco_balanced'
ANNOTS_DIR  = COCO_ROOT + '/annotations'
OUTPUT_DIR  = '/kaggle/working/fasterrcnn_output'
EPOCHS     = 150  # ← your request
BATCH_SIZE = 8      # keep
IMG_SIZE   = 256    # ← reduce from 320 to 256 (saves ~2 hrs)
PATIENCE   = 50     # ← half of 150
LR         = 0.005  # keep
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Auto-detect classes — only AP and MP
with open(f'{ANNOTS_DIR}/instances_train.json') as f: _d = json.load(f)
VALID_NAMES  = ['AP', 'MP']
valid_cats   = [c for c in _d['categories'] if c['name'] in VALID_NAMES]
valid_cats   = sorted(valid_cats, key=lambda x: x['id'])
CLASS_NAMES  = [c['name'] for c in valid_cats]
VALID_CAT_IDS= {c['id']: i+1 for i, c in enumerate(valid_cats)}  # 1-indexed for Faster R-CNN
NUM_CLASSES  = len(CLASS_NAMES) + 1  # +1 background

print('✅ Config OK')
print(f'   Classes     : {CLASS_NAMES}')
print(f'   NUM_CLASSES : {NUM_CLASSES}')
print(f'   Epochs      : {EPOCHS}')
print(f'   Batch size  : {BATCH_SIZE}')
print(f'   IMG_SIZE    : {IMG_SIZE}px (resized for speed)')
print(f'   Device      : {DEVICE}')
print()
for sp in ['train','val','test']:
    with open(f'{ANNOTS_DIR}/instances_{sp}.json') as f: dd=json.load(f)
    valid_anns=[a for a in dd['annotations'] if a['category_id'] in VALID_CAT_IDS]
    print(f'  ✅ {sp:5s} → {len(dd["images"])} images | {len(valid_anns)} valid anns')

In [ ]:
# ═══ Dataset ══════════════════════════════════════════════
class FastCocoDS(Dataset):
    def __init__(self, img_dir, ann_file, img_size=320):
        self.img_dir  = Path(img_dir)
        self.img_size = img_size
        with open(ann_file) as f: coco = json.load(f)
        abi = defaultdict(list)
        for a in coco['annotations']:
            if a['category_id'] in VALID_CAT_IDS:
                abi[a['image_id']].append(a)
        self.samples = []
        for img in coco['images']:
            p  = self.img_dir / img['file_name']
            if not p.exists(): continue
            va = [a for a in abi[img['id']] if a['bbox'][2]>=10 and a['bbox'][3]>=10]
            if va: self.samples.append((img, va))
        # Pre-compute scale factors for speed
        self.tfm = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
        ])
        print(f'  ✅ {len(self.samples)} valid images (AP+MP only)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        info, anns = self.samples[idx]
        img  = Image.open(self.img_dir/info['file_name']).convert('RGB')
        W0, H0 = img.size
        sx = self.img_size/W0; sy = self.img_size/H0
        img = self.tfm(img)
        boxes, labels = [], []
        for a in anns:
            x,y,w,h = a['bbox']
            x1=max(0.,x)*sx; y1=max(0.,y)*sy
            x2=min(float(W0),x+w)*sx; y2=min(float(H0),y+h)*sy
            if x2-x1>=5 and y2-y1>=5:
                boxes.append([x1,y1,x2,y2])
                labels.append(VALID_CAT_IDS[a['category_id']])
        if not boxes:
            boxes  = [[0.,0.,float(self.img_size),float(self.img_size)]]
            labels = [1]
        target = {
            'boxes'   : torch.tensor(boxes,  dtype=torch.float32),
            'labels'  : torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx]),
        }
        return img, target

def collate_fn(batch): return tuple(zip(*batch))

train_ds = FastCocoDS(f'{COCO_ROOT}/train',f'{ANNOTS_DIR}/instances_train.json',IMG_SIZE)
val_ds   = FastCocoDS(f'{COCO_ROOT}/val',  f'{ANNOTS_DIR}/instances_val.json',  IMG_SIZE)
test_ds  = FastCocoDS(f'{COCO_ROOT}/test', f'{ANNOTS_DIR}/instances_test.json', IMG_SIZE)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True, collate_fn=collate_fn,num_workers=4,pin_memory=True)
val_loader  =DataLoader(val_ds,  batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=4,pin_memory=True)
test_loader =DataLoader(test_ds, batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,num_workers=4,pin_memory=True)
print(f'\n✅ Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}')

In [ ]:
# ═══ Model — MobileNetV3 backbone (5x faster than ResNet50) ═
model = fasterrcnn_mobilenet_v3_large_fpn(
    weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
)
in_f  = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)
model = model.to(DEVICE)
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, momentum=0.9, weight_decay=5e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
print(f'✅ Faster R-CNN (MobileNetV3 backbone — fast!)')
print(f'   Parameters  : {sum(p.numel() for p in model.parameters()):,}')
print(f'   NUM_CLASSES : {NUM_CLASSES}')
print(f'   Note: MobileNetV3 is 5x faster than ResNet50 with similar accuracy')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Train 150 Epochs (Fast + Mixed Precision)
# ═══════════════════════════════════════════════════════════
import time
train_losses, val_losses = [], []
best_val   = float('inf')
best_pt    = os.path.join(OUTPUT_DIR, 'best.pt')
no_improve = 0
scaler     = torch.cuda.amp.GradScaler()

def run_epoch(model, loader, optimizer=None):
    model.train()
    total, count = 0, 0
    for imgs, tgts in loader:
        try:
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            bad  = any(
                t['boxes'].shape[0] == 0 or
                t['labels'].max() >= NUM_CLASSES or
                t['labels'].min() < 1
                for t in tgts
            )
            if bad: continue
            if optimizer:
                with torch.cuda.amp.autocast():
                    loss_dict = model(imgs, tgts)
                    loss = sum(loss_dict.values())
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                with torch.no_grad():
                    loss_dict = model(imgs, tgts)
                    loss = sum(loss_dict.values())
            total += loss.item()
            count += 1
        except Exception:
            continue
    return total / max(count, 1)

print('Starting Faster R-CNN Training (150 Epochs — Fast Mode)...')
print('='*65)
start = time.time()

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    tl = run_epoch(model, train_loader, optimizer)
    with torch.no_grad():
        vl = run_epoch(model, val_loader)
    scheduler.step()
    train_losses.append(tl)
    val_losses.append(vl)

    if vl < best_val:
        best_val   = vl
        torch.save(model.state_dict(), best_pt)
        no_improve = 0
        saved = '💾'
    else:
        no_improve += 1
        saved = ''

    ep_t = time.time() - t0
    eta  = ep_t * (EPOCHS - ep) / 60
    print(f'Epoch {ep:>3}/{EPOCHS} | train={tl:.4f} val={vl:.4f} best={best_val:.4f} | {ep_t:.0f}s | ETA:{eta:.0f}m {saved}')

    if no_improve >= PATIENCE:
        print(f'\n⏹️ Early stop at epoch {ep}')
        break

total_time = (time.time() - start) / 3600
print(f'\n✅ Training complete in {total_time:.1f} hrs')
print(f'   best.pt → {best_pt}')

In [ ]:
# ═══ Loss Curves ══════════════════════════════════════════
epochs_ran = len(train_losses)
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(range(1,epochs_ran+1),train_losses,label='Train',color='#2196F3',linewidth=1.8)
ax.plot(range(1,epochs_ran+1),val_losses,  label='Val',  color='#F44336',linewidth=1.8)
ax.set_title('Faster R-CNN — Esophageal Cancer — Loss Curves',fontweight='bold',fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True,alpha=0.3); ax.set_xlim(1,epochs_ran)
plt.tight_layout()
loss_path=os.path.join(OUTPUT_DIR,'fasterrcnn_loss_curves.png')
plt.savefig(loss_path,dpi=150,bbox_inches='tight'); plt.show()
pd.DataFrame({'epoch':range(1,epochs_ran+1),'train':train_losses,'val':val_losses}).to_csv(
    os.path.join(OUTPUT_DIR,'fasterrcnn_losses.csv'),index=False)
print(f'✅ Loss curves saved: {loss_path}')

In [ ]:
# ═══ Evaluate + Confusion Matrix ══════════════════════════
model.load_state_dict(torch.load(best_pt,map_location=DEVICE))
model.eval()
nc       = len(CLASS_NAMES)
cm_array = np.zeros((nc,nc),dtype=int)

def box_iou(a,b):
    xA=max(a[0],b[0]); yA=max(a[1],b[1])
    xB=min(a[2],b[2]); yB=min(a[3],b[3])
    i=max(0,xB-xA)*max(0,yB-yA)
    u=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-i
    return i/u if u>0 else 0

total_preds=0; total_gts=0
with torch.no_grad():
    for imgs,tgts in test_loader:
        imgs =[i.to(DEVICE) for i in imgs]
        preds=model(imgs)
        for pred,tgt in zip(preds,tgts):
            gb=tgt['boxes'].numpy(); gl=tgt['labels'].numpy()
            total_gts+=len(gb)
            keep=pred['scores'].cpu().numpy()>0.5
            pb=pred['boxes'].cpu().numpy()[keep]
            pl=pred['labels'].cpu().numpy()[keep]
            total_preds+=len(pb)
            matched=set()
            for pbox,plbl in zip(pb,pl):
                bi,bv=-1,0
                for gi,(gbox,glbl) in enumerate(zip(gb,gl)):
                    if gi in matched: continue
                    v=box_iou(pbox,gbox)
                    if v>bv: bv=v; bi=gi
                if bv>=0.5 and bi>=0:
                    gc=gl[bi]-1; pc=plbl-1
                    if 0<=gc<nc and 0<=pc<nc: cm_array[gc,pc]+=1
                    matched.add(bi)

print(f'✅ Evaluation done')
print(f'   Total GT        : {total_gts}')
print(f'   Total preds     : {total_preds}')
print(f'   CM matched      : {cm_array.sum()}')
print(f'   Confusion matrix:\n{cm_array}')

cm_n=cm_array.astype(float); rs=cm_n.sum(axis=1,keepdims=True); rs[rs==0]=1; cm_n/=rs
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('Faster R-CNN — Confusion Matrix — Esophageal Cancer',fontsize=13,fontweight='bold')
for ax,d,t,f in zip(axes,[cm_array,cm_n],['Raw Counts','Normalised'],['d','.2f']):
    sns.heatmap(d,annot=True,fmt=f,cmap='Blues',
                xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,
                ax=ax,linewidths=0.8,annot_kws={'size':13,'weight':'bold'})
    ax.set_title(t,fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fasterrcnn_confusion_matrix.png'),dpi=150,bbox_inches='tight')
plt.show(); print('✅ Confusion matrix saved')

In [ ]:
# ═══ Metrics Table ════════════════════════════════════════
def compute_metrics(cm,names):
    rows=[]; total=cm.sum()
    for i,cls in enumerate(names[:cm.shape[0]]):
        TP=cm[i,i]; FP=cm[:,i].sum()-TP; FN=cm[i,:].sum()-TP; TN=total-TP-FP-FN
        P=TP/(TP+FP) if TP+FP>0 else 0; R=TP/(TP+FN) if TP+FN>0 else 0
        F1=2*P*R/(P+R) if P+R>0 else 0; Acc=(TP+TN)/total if total>0 else 0
        rows.append({'Class':cls,'TP':int(TP),'FP':int(FP),'FN':int(FN),'TN':int(TN),
                     'Precision':round(P,4),'Recall':round(R,4),'F1-Score':round(F1,4),'Accuracy':round(Acc,4)})
    rows.append({'Class':'MACRO AVG','TP':'','FP':'','FN':'','TN':'',
                 'Precision':round(np.mean([r['Precision'] for r in rows]),4),
                 'Recall':round(np.mean([r['Recall'] for r in rows]),4),
                 'F1-Score':round(np.mean([r['F1-Score'] for r in rows]),4),
                 'Accuracy':round(np.mean([r['Accuracy'] for r in rows]),4)})
    return pd.DataFrame(rows)

mdf=compute_metrics(cm_array.astype(float),CLASS_NAMES)
print('='*65); print('  FASTER R-CNN METRICS'); print('='*65)
print(mdf.to_string(index=False)); print('='*65)

fig,ax=plt.subplots(figsize=(15,max(3,len(mdf)*0.65+1.2))); ax.axis('off')
tbl=ax.table(cellText=mdf.values.tolist(),colLabels=list(mdf.columns),cellLoc='center',loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2,1.8)
for j in range(len(mdf.columns)): tbl[0,j].set_facecolor('#1565C0'); tbl[0,j].set_text_props(color='white',fontweight='bold')
for i in range(1,len(mdf)):
    for j in range(len(mdf.columns)): tbl[i,j].set_facecolor('#F5F5F5' if i%2==0 else 'white')
last=len(mdf)
for j in range(len(mdf.columns)): tbl[last,j].set_facecolor('#BBDEFB'); tbl[last,j].set_text_props(fontweight='bold')
ax.set_title('Faster R-CNN — Precision/Recall/F1/Accuracy',fontsize=13,fontweight='bold',pad=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fasterrcnn_metrics_table.png'),dpi=150,bbox_inches='tight'); plt.show()
mdf.to_csv(os.path.join(OUTPUT_DIR,'fasterrcnn_metrics.csv'),index=False)
print('✅ Metrics saved')

In [ ]:
print('='*60); print('  FASTER R-CNN OUTPUT FILES'); print('='*60)
for name,path in [
    ('best.pt',          best_pt),
    ('Loss Curves PNG',  os.path.join(OUTPUT_DIR,'fasterrcnn_loss_curves.png')),
    ('Loss Curves CSV',  os.path.join(OUTPUT_DIR,'fasterrcnn_losses.csv')),
    ('Confusion Matrix', os.path.join(OUTPUT_DIR,'fasterrcnn_confusion_matrix.png')),
    ('Metrics Table PNG',os.path.join(OUTPUT_DIR,'fasterrcnn_metrics_table.png')),
    ('Metrics CSV',      os.path.join(OUTPUT_DIR,'fasterrcnn_metrics.csv')),
]:
    mark='✅' if os.path.exists(path) else '❌'
    print(f'  {mark}  {name:<22} → {path}')
print('\n🎉 Faster R-CNN Training Complete!')